# YOLO Segmentation + CLIP Leather Defect Detection Pipeline

This notebook implements a complete defect detection pipeline for leather products:
1. Train YOLO segmentation model on MVTec Leather dataset
2. Detect and segment leather surfaces
3. Segment defects within leather
4. Use CLIP to classify defect types

**Leather Defect Types:**
- **Color**: Color variations and discoloration stains
- **Cut**: Cuts, tears, and slashes in leather
- **Fold**: Folds, creases, and wrinkles
- **Glue**: Glue residue and adhesive marks
- **Poke**: Holes and punctures




## 1. Install Dependencies

In [ ]:
!pip install ultralytics opencv-python pillow torch torchvision tqdm
!pip install fastembed qdrant-edge-py


## 2. Import Libraries

In [ ]:
import os
import cv2
import numpy as np
import torch
from PIL import Image
from pathlib import Path
from ultralytics import YOLO
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm import tqdm
import yaml
import shutil
import uuid
from fastembed import ImageEmbedding, TextEmbedding
from qdrant_edge import (
    Distance,
    EdgeConfig,
    EdgeShard,
    EdgeVectorParams,
    Point,
    UpdateOperation,
    Query,
    QueryRequest
)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
%matplotlib inline



## 3. Prepare MVTec Dataset for YOLO Segmentation

In [ ]:
# 4. Download MVTec Leather Dataset
# Download from: https://www.mvtec.com/company/research/datasets/mvtec-ad
# Extract to: mvtec_anomaly_detection/leather/

In [ ]:
# Dataset paths
MVTEC_ROOT = "mvtec_anomaly_detection/leather"
YOLO_DATASET_ROOT = "yolo_leather_dataset"

# Create YOLO dataset structure
os.makedirs(f"{YOLO_DATASET_ROOT}/images/train", exist_ok=True)
os.makedirs(f"{YOLO_DATASET_ROOT}/images/val", exist_ok=True)
os.makedirs(f"{YOLO_DATASET_ROOT}/labels/train", exist_ok=True)
os.makedirs(f"{YOLO_DATASET_ROOT}/labels/val", exist_ok=True)

print("Dataset directories created")

In [ ]:
def prepare_yolo_dataset():
    """Prepare MVTec Leather dataset for DETECTION (not segmentation)"""
    
    defect_types = ['color', 'cut', 'fold', 'glue', 'poke']
    train_count = 0
    val_count = 0
    
    for defect_type in defect_types:
        test_img_dir = f"{MVTEC_ROOT}/test/{defect_type}"
        gt_dir = f"{MVTEC_ROOT}/ground_truth/{defect_type}"
        
        if not os.path.exists(test_img_dir):
            continue
        
        images = sorted([f for f in os.listdir(test_img_dir) if f.endswith('.png')])
        
        for idx, img_name in enumerate(images):
            img_path = f"{test_img_dir}/{img_name}"
            mask_name = img_name.replace('.png', '_mask.png')
            mask_path = f"{gt_dir}/{mask_name}"
            
            img = cv2.imread(img_path)
            if img is None:
                continue
            
            h, w = img.shape[:2]
            split = 'train' if idx % 5 != 0 else 'val'
            
            new_img_name = f"{defect_type}_{img_name}"
            shutil.copy(img_path, f"{YOLO_DATASET_ROOT}/images/{split}/{new_img_name}")
            
            label_path = f"{YOLO_DATASET_ROOT}/labels/{split}/{new_img_name.replace('.png', '.txt')}"
            
            with open(label_path, 'w') as f:
                f.write(f"0 0.5 0.5 1.0 1.0\n")  
                
                # Class 1: Defect - from mask
                if os.path.exists(mask_path):
                    defect_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
                    
                    if defect_mask is not None and np.sum(defect_mask) > 0:
                        # Get bounding box from mask
                        contours, _ = cv2.findContours(defect_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                        if len(contours) > 0:
                            largest = max(contours, key=cv2.contourArea)
                            x, y, bw, bh = cv2.boundingRect(largest)
                            
                            center_x = (x + bw/2) / w
                            center_y = (y + bh/2) / h
                            norm_w = bw / w
                            norm_h = bh / h
                            
                            f.write(f"1 {center_x:.6f} {center_y:.6f} {norm_w:.6f} {norm_h:.6f}\n")
            
            if split == 'train':
                train_count += 1
            else:
                val_count += 1
    
    return train_count, val_count



train_count, val_count = prepare_yolo_dataset()


In [ ]:
def create_leather_mask(image):
    """
    Create leather surface mask using texture-based segmentation.
    Leather typically has uniform texture and color across the surface.
    """
    h, w = image.shape[:2]
    
    # Convert to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    # Leather is typically in mid-range intensity (not too dark, not too bright)
    # Remove very bright background (white/light gray)
    _, mask_bright = cv2.threshold(gray, 240, 255, cv2.THRESH_BINARY_INV)
    

    _, mask_dark = cv2.threshold(gray, 15, 255, cv2.THRESH_BINARY)
    
    leather_mask = cv2.bitwise_and(mask_bright, mask_dark)
    
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (20, 20))
    leather_mask = cv2.morphologyEx(leather_mask, cv2.MORPH_CLOSE, kernel, iterations=3)
    leather_mask = cv2.morphologyEx(leather_mask, cv2.MORPH_OPEN, kernel, iterations=2)
    
    contours, hierarchy = cv2.findContours(leather_mask, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)
    for i in range(len(contours)):
        cv2.drawContours(leather_mask, contours, i, 255, -1)
    
    contours, _ = cv2.findContours(leather_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if len(contours) == 0:
        return None
    
    image_area = h * w
    valid_contours = []
    
    for contour in contours:
        area = cv2.contourArea(contour)
        
        if 0.30 * image_area < area < 0.90 * image_area:
            valid_contours.append(contour)
    
    if len(valid_contours) == 0:
        largest = max(contours, key=cv2.contourArea)
        valid_contours = [largest]
    
    final_mask = np.zeros_like(gray)
    cv2.drawContours(final_mask, valid_contours, -1, 255, -1)
    
    return final_mask



In [ ]:
# Create YOLO dataset configuration file
dataset_config = {
    'path': os.path.abspath(YOLO_DATASET_ROOT),
    'train': 'images/train',
    'val': 'images/val',
    'names': {
        0: 'leather',
        1: 'defect'
    }
}


config_path = f"{YOLO_DATASET_ROOT}/dataset.yaml"
with open(config_path, 'w') as f:
    yaml.dump(dataset_config, f)

print(f"Dataset config saved to {config_path}")
print("\nConfig contents:")
print(yaml.dump(dataset_config, default_flow_style=False))

## 4. Train YOLO Segmentation Model

In [ ]:
model = YOLO('yolov8n.pt') # Use nano model for faster training

# Train the model with better parameters
results = model.train(
    data=config_path,
    epochs=100,  
    imgsz=640,
    batch=8,
    name='leather_defect_seg',
    patience=20, 
    save=True,
    device=device,
    plots=True,
    
    # Data augmentation to improve generalization
    hsv_h=0.015,  
    hsv_s=0.7,    
    hsv_v=0.4,    
    degrees=10, 
    translate=0.1, 
    scale=0.2,   
    flipud=0.5,  
    fliplr=0.5,   
    mosaic=1.0,  
    
    # Optimization
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    
    # Loss weights for segmentation model
    # box=7.5,
    # cls=0.5,
    # dfl=1.5
)

print("\nTraining completed!")

print("\nTraining completed!")

## 5. Load Trained Model and CLIP

In [ ]:
# Load trained YOLO model
trained_model_path = 'runs/detect/leather_defect_seg/weights/best.pt'
yolo_model = YOLO(trained_model_path)
print(f"Loaded YOLO model from {trained_model_path}")

# Setup Qdrant Edge and FastEmbed
TEXT_MODEL_NAME = 'Qdrant/clip-ViT-B-32-text'
VISION_MODEL_NAME = 'Qdrant/clip-ViT-B-32-vision'
MODELS_DIR = "./qdrant-edge-directory/models"
SHARD_DIRECTORY = "./qdrant-edge-directory/shard"
VECTOR_DIMENSION = 512
VECTOR_NAME = "defect-vector"

# Download and cache models (do this once during setup)
print("Downloading FastEmbed models...")
ImageEmbedding(
    model_name=VISION_MODEL_NAME,
    cache_dir=MODELS_DIR
)

TextEmbedding(
    model_name=TEXT_MODEL_NAME,
    cache_dir=MODELS_DIR
)
print(f"Models cached in {MODELS_DIR}")

# Initialize Edge Shard
Path(SHARD_DIRECTORY).mkdir(parents=True, exist_ok=True)
config = EdgeConfig(
    vectors={
        VECTOR_NAME: EdgeVectorParams(
            size=VECTOR_DIMENSION,
            distance=Distance.Cosine,
        )
    }
)

edge_shard = EdgeShard.create(SHARD_DIRECTORY, config)
print("Qdrant Edge Shard initialized")

# Initialize embedding models
image_model = ImageEmbedding(
    model_name=VISION_MODEL_NAME,
    cache_dir=MODELS_DIR,
    local_files_only=True
)

text_model = TextEmbedding(
    model_name=TEXT_MODEL_NAME,
    cache_dir=MODELS_DIR,
    local_files_only=True
)
print("FastEmbed models loaded")


## 6. Define Defect Classification with CLIP

In [ ]:
## 6. Image-Based Defect Classification with Qdrant Edge

import glob

print("FastEmbed image model loaded")


def extract_defect_crop_from_mask(image, mask, padding_percent=7):
    """
    Extract defect region from image using ground truth mask with minimal padding
    
    Args:
        image: Original image (numpy array)
        mask: Binary mask (numpy array)
        padding_percent: Percentage of bbox size to add as padding (5-10%)
    
    Returns:
        Cropped defect region with padding
    """
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if len(contours) == 0:
        return None
    
    largest_contour = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest_contour)
    
    pad_x = int(w * padding_percent / 100)
    pad_y = int(h * padding_percent / 100)
    
    x1 = max(0, x - pad_x)
    y1 = max(0, y - pad_y)
    x2 = min(image.shape[1], x + w + pad_x)
    y2 = min(image.shape[0], y + h + pad_y)
    
    defect_crop = image[y1:y2, x1:x2]
    
    return defect_crop


# Build image embedding database from cropped defect regions
print("\n" + "="*70)
print("Building Image Embedding Database from Cropped Defect Regions")
print("="*70)

defect_types = ['color', 'cut', 'fold', 'glue', 'poke']
image_points = []
total_images = 0
skipped_images = 0

for defect_type in defect_types:
    test_img_dir = f"{MVTEC_ROOT}/test/{defect_type}"
    gt_dir = f"{MVTEC_ROOT}/ground_truth/{defect_type}"
    
    if not os.path.exists(test_img_dir) or not os.path.exists(gt_dir):
        print(f"Skipping {defect_type} - directory not found")
        continue
    
    image_files = sorted([f for f in os.listdir(test_img_dir) if f.endswith('.png')])[:20]  # 20 examples per type
    
    print(f"\nProcessing {defect_type} defects...")
    
    for img_name in tqdm(image_files, desc=f"  Embedding {defect_type}"):
        img_path = f"{test_img_dir}/{img_name}"
        mask_name = img_name.replace('.png', '_mask.png')
        mask_path = f"{gt_dir}/{mask_name}"
        
        # Check if mask exists
        if not os.path.exists(mask_path):
            skipped_images += 1
            continue
        
        try:
            image = cv2.imread(img_path)
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            
            if image is None or mask is None:
                skipped_images += 1
                continue
            
            defect_crop = extract_defect_crop_from_mask(image, mask, padding_percent=7)
            
            if defect_crop is None or defect_crop.size == 0:
                skipped_images += 1
                continue
            
            temp_crop_path = f"/tmp/defect_crop_{uuid.uuid4().hex[:8]}.jpg"
            cv2.imwrite(temp_crop_path, defect_crop)
            
            embeddings = list(image_model.embed([Path(temp_crop_path)]))[0]
            
            os.remove(temp_crop_path)
            
            point = Point(
                id=str(uuid.uuid4()),
                vector={VECTOR_NAME: embeddings.tolist()},
                payload={
                    "defect_type": defect_type,
                    "image_path": img_path,
                    "mask_path": mask_path,
                    "filename": img_name,
                    "crop_method": "ground_truth_mask",
                    "padding_percent": 7
                }
            )
            
            image_points.append(point)
            total_images += 1
            
        except Exception as e:
            print(f" Failed to process {img_name}: {e}")
            skipped_images += 1
    
    print(f"Added {len([p for p in image_points if p.payload['defect_type'] == defect_type])} cropped defects for {defect_type}")

print(f"\n{'='*70}")
print(f"Defect Crop Embedding Database Ready")
print(f"{'='*70}")
print(f"Total defect crops indexed: {total_images}")
print(f"Skipped images: {skipped_images}")


print("\n" + "="*70)
print("Building Leather Embedding Database from Good Samples")
print("="*70)

good_leather_dir = f"{MVTEC_ROOT}/train/good"
leather_count = 0

if os.path.exists(good_leather_dir):
    good_images = sorted([f for f in os.listdir(good_leather_dir) if f.endswith('.png')])[:30]  # 30 good examples
    
    print(f"\nProcessing good leather samples...")
    
    for img_name in tqdm(good_images, desc="  Embedding good leather"):
        img_path = f"{good_leather_dir}/{img_name}"
        
        try:
            image = cv2.imread(img_path)
            
            if image is None:
                continue
            
            leather_mask = create_leather_mask(image)
            
            if leather_mask is None:
                h, w = image.shape[:2]
                margin = int(min(h, w) * 0.1)
                leather_crop = image[margin:h-margin, margin:w-margin]
            else:
                leather_crop = extract_defect_crop_from_mask(image, leather_mask, padding_percent=5)
            
            if leather_crop is None or leather_crop.size == 0:
                continue
            
            temp_crop_path = f"/tmp/leather_crop_{uuid.uuid4().hex[:8]}.jpg"
            cv2.imwrite(temp_crop_path, leather_crop)
            
            embeddings = list(image_model.embed([Path(temp_crop_path)]))[0]
            
            os.remove(temp_crop_path)
            
            point = Point(
                id=str(uuid.uuid4()),
                vector={VECTOR_NAME: embeddings.tolist()},
                payload={
                    "defect_type": "good_leather",  # Special type for good leather
                    "image_path": img_path,
                    "filename": img_name,
                    "crop_method": "leather_mask",
                    "padding_percent": 5
                }
            )
            
            image_points.append(point)
            leather_count += 1
            
        except Exception as e:
            print(f"Failed to process {img_name}: {e}")
    
    print(f"Added {leather_count} good leather samples")
else:
    print(f"Good leather directory not found: {good_leather_dir}")


# Store ALL embeddings in Edge Shard (defects + leather)
print(f"\n{'='*70}")
print(f"Storing embeddings in Qdrant Edge...")
print(f"{'='*70}")

edge_shard.update(UpdateOperation.upsert_points(image_points))

print(f"\n Complete Embedding Database Stored")
print(f"Total embeddings: {len(image_points)}")
print(f"  - Defect crops: {total_images}")
print(f"  - Good leather: {leather_count}")
print(f"  - Defect types: {defect_types}")
print(f"  - Crops per defect type: ~20")
print(f"  - Vector dimension: {VECTOR_DIMENSION}")
print(f"  - Distance metric: Cosine")


print(f"\nVerifying embeddings...")
test_query = [0.0] * VECTOR_DIMENSION
results = edge_shard.query(
    QueryRequest(
        query=Query.Nearest(test_query, using=VECTOR_NAME),
        limit=200,
        with_payload=True
    )
)

from collections import Counter
types = [r.payload.get('defect_type', 'unknown') for r in results]
type_counts = Counter(types)

print(f"\n Verification Complete")
print(f"Total embeddings in shard: {len(results)}")
print(f"Breakdown by type:")
for defect_type, count in sorted(type_counts.items()):
    print(f"  - {defect_type}: {count}")


# Enhanced Image-to-Image Classification Function
def classify_with_fastembed(image_crop, category_type="defect", top_k=15):
    """
    Classify using pure image-to-image matching with Qdrant Edge
    Supports both leather and defect classification
    
    Args:
        image_crop: Cropped region (numpy array)
        category_type: "leather" or "defect"
        top_k: Number of nearest neighbors to retrieve
    
    Returns:
        List of classification results with confidence scores
    """
    image_pil = Image.fromarray(cv2.cvtColor(image_crop, cv2.COLOR_BGR2RGB))
    
    temp_path = f"/tmp/query_{category_type}_{uuid.uuid4().hex[:8]}.jpg"
    image_pil.save(temp_path)
    
    try:
        query_embedding = list(image_model.embed([Path(temp_path)]))[0]
        
        results = edge_shard.query(
            QueryRequest(
                query=Query.Nearest(query_embedding.tolist(), using=VECTOR_NAME),
                limit=top_k * 2,
                with_vector=False,
                with_payload=True
            )
        )
        
        if category_type == "leather":
            good_leather_count = 0
            defect_count = 0
            
            for result in results[:top_k]:
                defect_type = result.payload['defect_type']
                if defect_type == 'good_leather':
                    good_leather_count += 1
                else:
                    defect_count += 1
            
            total = good_leather_count + defect_count
            if total == 0:
                return [{'category': 'unknown', 'confidence': 0.0}]
            
            classifications = [
                {
                    'category': 'good leather without defects',
                    'confidence': float(good_leather_count / total),
                    'votes': good_leather_count
                },
                {
                    'category': 'defective leather with damage',
                    'confidence': float(defect_count / total),
                    'votes': defect_count
                }
            ]
            
            classifications.sort(key=lambda x: x['confidence'], reverse=True)
            return classifications
            
        else:  
            defect_votes = {}
            defect_scores = {}
            defect_distances = {}
            
            for result in results:
                defect_type = result.payload['defect_type']
                
                if defect_type == 'good_leather':
                    continue
                
                similarity = 1 - result.score  
                distance = result.score
                
                if defect_type not in defect_votes:
                    defect_votes[defect_type] = 0
                    defect_scores[defect_type] = []
                    defect_distances[defect_type] = []
                
                defect_votes[defect_type] += 1
                defect_scores[defect_type].append(similarity)
                defect_distances[defect_type].append(distance)
            
            classifications = []
            total_votes = sum(defect_votes.values())
            
            if total_votes == 0:
                return [{'category': 'unknown', 'confidence': 0.0}]
            
            for defect_type, votes in defect_votes.items():
                vote_ratio = votes / total_votes
                avg_similarity = np.mean(defect_scores[defect_type])
                min_distance = np.min(defect_distances[defect_type])
                
                confidence = 0.7 * avg_similarity + 0.3 * vote_ratio
                
                classifications.append({
                    'category': defect_type,
                    'confidence': float(confidence),
                    'votes': votes,
                    'avg_similarity': float(avg_similarity),
                    'min_distance': float(min_distance)
                })
            
            classifications.sort(key=lambda x: x['confidence'], reverse=True)
            
            return classifications
        
    finally:
        if os.path.exists(temp_path):
            os.remove(temp_path)


print("\n" + "="*70)
print("Image-to-Image Classifier Ready")
print("="*70)
print("Classification method: Pure image-to-image matching")
print("Supports:")
print("  - Leather classification (good vs defective)")
print("  - Defect type classification (color, cut, fold, glue, poke)")
print("Database: Ground truth masked crops with 5-7% padding")


## 7. Complete Detection Pipeline

In [ ]:
def merge_overlapping_detections(detections, iou_threshold=0.5):
    """Merge overlapping detections using NMS"""
    if len(detections) <= 1:
        return detections
    
    boxes = np.array([d['bbox'] for d in detections])
    scores = np.array([d['yolo_confidence'] for d in detections])
    
    x1 = boxes[:, 0]
    y1 = boxes[:, 1]
    x2 = boxes[:, 2]
    y2 = boxes[:, 3]
    
    areas = (x2 - x1) * (y2 - y1)
    
    order = scores.argsort()[::-1]
    
    keep = []
    while order.size > 0:
        i = order[0]
        keep.append(i)
        
        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])
        
        w = np.maximum(0, xx2 - xx1)
        h = np.maximum(0, yy2 - yy1)
        
        intersection = w * h
        iou = intersection / (areas[i] + areas[order[1:]] - intersection)
        
        inds = np.where(iou <= iou_threshold)[0]
        order = order[inds + 1]
    
    return [detections[i] for i in keep]
def detect_and_classify_defects(image_path, conf_threshold=0.15, iou_threshold=0.7):
    """
    Detection-only pipeline (no segmentation masks)
    """
    image = cv2.imread(image_path)
    if image is None:
        print(f"Error: Could not read image {image_path}")
        return None
    
    original_image = image.copy()
    
    results = yolo_model(
        image, 
        conf=conf_threshold, 
        iou=iou_threshold,
        verbose=False
    )
    
    defects = []
    
    for result in results:

        
        boxes = result.boxes.xyxy.cpu().numpy()
        confidences = result.boxes.conf.cpu().numpy()
        classes = result.boxes.cls.cpu().numpy()
        
        print(f"DEBUG: Found {len(boxes)} total detections")
        
        for box, conf, cls in zip(boxes, confidences, classes):
            print(f"DEBUG: class={int(cls)}, conf={conf:.3f}")
            
            if int(cls) != 1:
                continue
            
            x1, y1, x2, y2 = map(int, box)
            
            box_area = (x2 - x1) * (y2 - y1)
            image_area = image.shape[0] * image.shape[1]
            area_ratio = box_area / image_area
            
            if area_ratio < 0.001 or area_ratio > 0.5:
                print(f"DEBUG: Skipped - area_ratio={area_ratio:.4f}")
                continue
            
            crop = original_image[y1:y2, x1:x2]
            if crop.size == 0:
                continue
            
            classification = classify_with_fastembed(crop, category_type="defect", top_k=15)
            
            mask = np.zeros((image.shape[0], image.shape[1]), dtype=np.float32)
            mask[y1:y2, x1:x2] = 1.0
            
            defects.append({
                'bbox': (x1, y1, x2, y2),
                'yolo_confidence': float(conf),
                'mask': mask,  # Simple box mask for visualization
                'area_ratio': area_ratio,
                'clip_classification': classification[0] if classification else {'category': 'unknown', 'confidence': 0.0},
                'all_clip_results': classification
            })
    
    if len(defects) > 0:
        defects = merge_overlapping_detections(defects, iou_threshold=0.5)
    
    print(f"Detected {len(defects)} defect(s)")
    
    return {
        'image': original_image,
        'defects': defects
    }

def visualize_defects(result_dict, save_path=None):
    """Visualize defect detection results only"""
    if result_dict is None:
        print("No results to visualize")
        return
    
    image = result_dict['image'].copy()
    defects = result_dict.get('defects', [])
    
    overlay = image.copy()
    img_h, img_w = image.shape[:2]
    
    for defect in defects:
        x1, y1, x2, y2 = defect['bbox']
        mask = defect['mask']
        clip_result = defect['clip_classification']
        
        mask_resized = cv2.resize(mask, (img_w, img_h))
        color = (0, 0, 255)  # Red for defects
        overlay[mask_resized > 0.5] = color
        
        cv2.rectangle(image, (x1, y1), (x2, y2), color, 3)
        
        defect_type = clip_result['category']
        confidence = clip_result['confidence']
        
        votes = clip_result.get('votes', 0)
        if votes > 0:
            label = f"{defect_type}: {confidence:.2f} ({votes}v)"
        else:
            label = f"{defect_type}: {confidence:.2f}"
        
        cv2.putText(image, label, (x1, y1 - 10), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    
    result_image = cv2.addWeighted(image, 0.7, overlay, 0.3, 0)
    
    status_text = f"Defects: {len(defects)}"
    cv2.putText(result_image, status_text, (10, 40), 
               cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)
    
    result_image_rgb = cv2.cvtColor(result_image, cv2.COLOR_BGR2RGB)
    
    fig, ax = plt.subplots(figsize=(15, 10))
    ax.imshow(result_image_rgb)
    ax.axis('off')
    
    title = f"Detected {len(defects)} defect(s)"
    ax.set_title(title, fontsize=18, fontweight='bold')
    
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=150)
        print(f"Saved visualization to {save_path}")
    
    plt.tight_layout()
    plt.show()
    plt.close(fig)
    
    print(f"\n{'='*70}")
    print(f"Detection Results:")
    print(f"  Total Defects: {len(defects)}")
    print(f"{'='*70}")
    
    for idx, defect in enumerate(defects, 1):
        print(f"\nDefect #{idx}:")
        print(f"  YOLO Confidence: {defect['yolo_confidence']:.3f}")
        print(f"  Area Ratio: {defect['area_ratio']:.1%}")
        print(f"  Bounding Box: {defect['bbox']}")
        print(f"  CLIP Classification:")
        for result in defect['all_clip_results'][:5]:
            votes = result.get('votes', 0)
            avg_sim = result.get('avg_similarity', 0)
            print(f"    - {result['category']:5s}: {result['confidence']:.3f} (votes={votes}, sim={avg_sim:.3f})")


## 8. Test on Sample Images

In [ ]:



# Test images - one from each defect type
test_images = [
    f"{MVTEC_ROOT}/test/color/005.png",
    f"{MVTEC_ROOT}/test/cut/005.png",
    f"{MVTEC_ROOT}/test/fold/005.png",
    f"{MVTEC_ROOT}/test/glue/005.png",
    f"{MVTEC_ROOT}/test/poke/005.png"
]


for img_path in test_images:
    if os.path.exists(img_path):
        print(f"\n{'='*70}")
        print(f"Processing: {os.path.basename(img_path)}")
        print(f"Expected defect type: {os.path.basename(os.path.dirname(img_path))}")
        print(f"{'='*70}")
        
        # Detect and classify defects
        result = detect_and_classify_defects(
            img_path, 
            conf_threshold=0.15,  # Low threshold to catch all defects
            iou_threshold=0.9
        )
        
        if result is None:
            print("Failed to process image")
            continue
        
        visualize_defects(result)
        
    else:
        print(f"Image not found: {img_path}")


## 12. Export Model for Production

In [ ]:
## 12. Export YOLO Model for DeepStream

import os

print("\n" + "="*70)
print("EXPORTING YOLO MODEL FOR DEEPSTREAM")
print("="*70)

print("\n1. Exporting to ONNX...")
onnx_path = yolo_model.export(format='onnx', simplify=True, dynamic=False, imgsz=640)
print(f"ONNX model: {onnx_path}")

# Export to TensorRT engine (optimized for GPU)
print("\n2. Exporting to TensorRT engine...")
engine_path = yolo_model.export(
    format='engine',
    device=0,  
    half=True,  
    workspace=4,
    imgsz=640
)
print(f"TensorRT engine: {engine_path}")

model_info = {
    'onnx_path': str(onnx_path),
    'engine_path': str(engine_path),
    'input_size': 640,
    'num_classes': 2,
    'class_names': ['leather', 'defect'],
    'confidence_threshold': 0.15,
    'iou_threshold': 0.7
}

import json
with open('deepstream_model_info.json', 'w') as f:
    json.dump(model_info, f, indent=2)

print(f"\n Model info saved to: deepstream_model_info.json")
print("\nModel files ready for DeepStream:")
print(f"  - ONNX: {onnx_path}")
print(f"  - TensorRT: {engine_path}")
